# CIRNAC/ISC First Nations Profile Scraping

Scrape official First Nations profile pages from ISC (Indigenous Services Canada).
Each community has a predictable URL using its INAC band number.

**Data available:** name, address, postal code, phone, fax, website URL, tribal council, census data.

Data flows: ISC profiles -> this notebook (CSV/JSON) -> NC source-manager import -> Minoo sync.

## Prerequisites
- NC source-manager API at `api.northcloud.one` (for community list with band numbers)
- `pip install -e '.[notebooks]'` from indigenous-harvesters root

In [ ]:
import httpx
import pandas as pd
import json
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from tqdm.notebook import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

NC_BASE_URL = "https://api.northcloud.one"
ISC_PROFILE_URL = "https://fnp-ppn.aadnc-aandc.gc.ca/fnp/Main/Search/FNMain.aspx?BAND_NUMBER={band_number}"
REQUEST_DELAY = 1.0  # polite delay between ISC requests
REQUEST_TIMEOUT = 15.0

## 1. Fetch all communities from NC

Get the full list of 637 First Nations with their INAC band numbers.

In [ ]:
def fetch_all_communities() -> list[dict]:
    """Fetch all First Nations communities from NC source-manager."""
    all_communities: list[dict] = []
    offset = 0
    limit = 200
    while True:
        resp = httpx.get(
            f"{NC_BASE_URL}/api/v1/communities",
            params={"limit": limit, "offset": offset, "type": "first_nation"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        communities = data.get("communities", []) or []
        all_communities.extend(communities)
        if len(communities) < limit:
            break
        offset += limit
    print(f"Fetched {len(all_communities)} communities")
    return all_communities

communities = fetch_all_communities()
df_communities = pd.DataFrame(communities)

# Filter to those with INAC band numbers
with_inac = df_communities[df_communities["inac_id"].notna() & (df_communities["inac_id"] != "")]
print(f"Communities with INAC band number: {len(with_inac)}")
with_inac[["name", "province", "inac_id"]].head(10)

## 2. ISC Profile Parser

Parse the ASP.NET span-based profile page. Field IDs follow pattern `plcMain_txt{FieldName}`.

In [ ]:
@dataclass
class ISCProfile:
    band_number: str
    community_id: str
    official_name: str = ""
    phonetic_spelling: str = ""
    address: str = ""
    postal_code: str = ""
    phone: str = ""
    fax: str = ""
    website: str = ""
    tribal_council: str = ""
    # Census fields (from General Information sub-page if available)
    registered_pop: int = 0
    on_reserve_pop: int = 0
    off_reserve_pop: int = 0
    land_area_hectares: float = 0.0
    reserve_names: str = ""
    error: str = ""
    scraped_at: str = ""


def get_span_text(soup: BeautifulSoup, span_id: str) -> str:
    """Extract text from an ASP.NET span by ID."""
    el = soup.find("span", id=span_id)
    if el:
        return el.get_text(strip=True)
    return ""


def get_link_url(soup: BeautifulSoup, label_span_id: str) -> str:
    """Find a link URL near a label span.

    ISC layout: label is in <div class="col-md-2">, link is in the next
    sibling <div>. Use find_all_next on the label to search forward in DOM.
    """
    label = soup.find("span", id=label_span_id)
    if not label:
        return ""
    # Search forward from the label for the first external link
    for el in label.find_all_next("a", href=True, limit=3):
        href = el["href"]
        if href.startswith("http") and "gc.ca" not in href and "canada.ca" not in href:
            return href
    return ""


def parse_isc_profile(html: str, band_number: str, community_id: str) -> ISCProfile:
    """Parse an ISC First Nations profile page."""
    profile = ISCProfile(
        band_number=band_number,
        community_id=community_id,
        scraped_at=datetime.now(timezone.utc).isoformat(),
    )

    soup = BeautifulSoup(html, "lxml")

    profile.official_name = get_span_text(soup, "plcMain_txtBandName")
    profile.phonetic_spelling = get_span_text(soup, "plcMain_txtPhoneticSpelling")

    # Address field includes postal code on ISC (e.g. "978 TASHMOO AVE, SARNIA, ON, N7T 7H5")
    # Extract postal code from the address string if txtPostalCode is empty
    raw_address = get_span_text(soup, "plcMain_txtAddress")
    profile.postal_code = get_span_text(soup, "plcMain_txtPostalCode")

    # If postal code is embedded in address, extract and clean
    if not profile.postal_code and raw_address:
        pc_match = re.search(r"[A-Z]\d[A-Z]\s?\d[A-Z]\d", raw_address)
        if pc_match:
            profile.postal_code = pc_match.group()
            # Remove postal code from address
            raw_address = raw_address[:pc_match.start()].rstrip(", ")

    profile.address = raw_address
    profile.phone = get_span_text(soup, "plcMain_txtPhone")
    profile.fax = get_span_text(soup, "plcMain_txtFax")
    profile.website = get_link_url(soup, "plcMain_lblWebSite")

    # Tribal council — find the link after the TC heading
    tc_heading = soup.find("span", id="plcMain_lblTCTitle")
    if tc_heading:
        tc_link = tc_heading.find_next("a")
        if tc_link:
            profile.tribal_council = tc_link.get_text(strip=True)

    if not profile.official_name:
        profile.error = "no_data_found"

    return profile

print("ISC profile parser loaded.")

## 3. Scrape ISC Profiles

Fetch each community's ISC profile page using their band number.

In [ ]:
def scrape_isc_profile(client: httpx.Client, community: dict) -> ISCProfile:
    """Fetch and parse an ISC profile page for a community."""
    band_number = str(community.get("inac_id", ""))
    community_id = community["id"]

    if not band_number:
        return ISCProfile(
            band_number="",
            community_id=community_id,
            error="no_band_number",
            scraped_at=datetime.now(timezone.utc).isoformat(),
        )

    url = ISC_PROFILE_URL.format(band_number=band_number)

    try:
        resp = client.get(url, timeout=REQUEST_TIMEOUT, follow_redirects=True)
        resp.raise_for_status()
        return parse_isc_profile(resp.text, band_number, community_id)
    except Exception as e:
        return ISCProfile(
            band_number=band_number,
            community_id=community_id,
            error=str(e)[:100],
            scraped_at=datetime.now(timezone.utc).isoformat(),
        )

print("Scraper loaded.")

In [ ]:
# Limit for testing — set to None to scrape all 637
LIMIT = 5

targets = with_inac.to_dict("records")
if LIMIT:
    targets = targets[:LIMIT]
print(f"Scraping {len(targets)} ISC profiles...")

profiles: list[ISCProfile] = []

with httpx.Client(
    headers={"User-Agent": "MinooCommunityEnricher/0.1 (+https://minoo.app; Indigenous data enrichment)"},
    follow_redirects=True,
) as client:
    for community in tqdm(targets, desc="ISC Profiles"):
        profile = scrape_isc_profile(client, community)
        profiles.append(profile)
        time.sleep(REQUEST_DELAY)

print(f"\nDone. {len(profiles)} profiles scraped.")
print(f"  Errors: {sum(1 for p in profiles if p.error)}")
print(f"  With phone: {sum(1 for p in profiles if p.phone)}")
print(f"  With website: {sum(1 for p in profiles if p.website)}")
print(f"  With address: {sum(1 for p in profiles if p.address)}")

## 4. Assemble Dataset

In [ ]:
# Build enrichment DataFrame
rows = [asdict(p) for p in profiles]
df_profiles = pd.DataFrame(rows)

print(f"=== ISC PROFILE ENRICHMENT SUMMARY ===")
print(f"Total scraped: {len(df_profiles)}")
print(f"Successful: {(~df_profiles['error'].astype(bool)).sum()}")
print(f"\nField coverage:")
for col in ["official_name", "address", "postal_code", "phone", "fax", "website", "tribal_council", "phonetic_spelling"]:
    count = df_profiles[col].astype(bool).sum()
    pct = count / len(df_profiles) * 100 if len(df_profiles) > 0 else 0
    print(f"  {col}: {count}/{len(df_profiles)} ({pct:.0f}%)")

display(df_profiles[["band_number", "official_name", "address", "postal_code", "phone", "fax", "website", "tribal_council", "error"]].head(20))

## 5. Parse Address Components

Split the ISC address string (e.g. `978 TASHMOO AVENUE, SARNIA, ON`) into street, city, province.

In [ ]:
PROVINCES = {"AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT"}

def parse_address(address: str) -> dict:
    """Parse ISC address format: 'STREET, CITY, PROVINCE' or variations."""
    result = {"street": "", "city": "", "province": ""}
    if not address:
        return result

    parts = [p.strip() for p in address.split(",")]

    if len(parts) >= 3:
        # Check if last part is a province code
        if parts[-1].upper() in PROVINCES:
            result["province"] = parts[-1].upper()
            result["city"] = parts[-2].title()
            result["street"] = ", ".join(parts[:-2]).title()
        else:
            result["street"] = parts[0].title()
            result["city"] = parts[1].title()
            result["province"] = parts[2].upper()
    elif len(parts) == 2:
        if parts[-1].upper() in PROVINCES:
            result["city"] = parts[0].title()
            result["province"] = parts[-1].upper()
        else:
            result["street"] = parts[0].title()
            result["city"] = parts[1].title()
    else:
        result["street"] = address.title()

    return result

# Apply address parsing
if not df_profiles.empty:
    addr_parsed = df_profiles["address"].apply(parse_address).apply(pd.Series)
    df_profiles["street"] = addr_parsed["street"]
    df_profiles["city"] = addr_parsed["city"]
    df_profiles["addr_province"] = addr_parsed["province"]

    print("Address parsing complete.")
    display(df_profiles[["official_name", "street", "city", "addr_province", "postal_code"]].head(10))

## 6. Export Dataset

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# CSV export
csv_path = DATA_DIR / f"isc_profiles_{timestamp}.csv"
df_profiles.to_csv(csv_path, index=False)
print(f"CSV: {csv_path} ({len(df_profiles)} rows)")

# JSON export (for NC import)
json_export = [asdict(p) for p in profiles if not p.error]
json_path = DATA_DIR / f"isc_profiles_{timestamp}.json"
json_path.write_text(json.dumps(json_export, indent=2))
print(f"JSON (clean): {json_path} ({len(json_export)} rows)")

# Website URLs only (for notebook 01)
websites = df_profiles[df_profiles["website"].astype(bool)][["community_id", "band_number", "official_name", "website"]]
if not websites.empty:
    websites_path = DATA_DIR / f"community_websites_{timestamp}.csv"
    websites.to_csv(websites_path, index=False)
    print(f"Websites: {websites_path} ({len(websites)} communities with URLs)")

## 7. Data Quality Review

In [ ]:
if not df_profiles.empty:
    print("=== DATA QUALITY ===")

    # Check for errors
    errors = df_profiles[df_profiles["error"].astype(bool)]
    if not errors.empty:
        print(f"\nErrors ({len(errors)}):")
        print(errors[["band_number", "error"]].to_string(index=False))

    # Validate phone format
    phone_pattern = re.compile(r"^\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$")
    has_phone = df_profiles[df_profiles["phone"].astype(bool)]
    bad_phones = has_phone[~has_phone["phone"].str.match(r"^\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$", na=False)]
    if not bad_phones.empty:
        print(f"\nWARN: {len(bad_phones)} non-standard phone formats:")
        print(bad_phones[["official_name", "phone"]].head(10).to_string(index=False))

    # Validate postal codes
    has_pc = df_profiles[df_profiles["postal_code"].astype(bool)]
    bad_pc = has_pc[~has_pc["postal_code"].str.match(r"^[A-Z]\d[A-Z]\s?\d[A-Z]\d$", na=False)]
    if not bad_pc.empty:
        print(f"\nWARN: {len(bad_pc)} non-standard postal codes:")
        print(bad_pc[["official_name", "postal_code"]].head(10).to_string(index=False))

    # Website URL validation
    has_url = df_profiles[df_profiles["website"].astype(bool)]
    bad_urls = has_url[~has_url["website"].str.startswith("http", na=False)]
    if not bad_urls.empty:
        print(f"\nWARN: {len(bad_urls)} non-HTTP website URLs:")
        print(bad_urls[["official_name", "website"]].head(10).to_string(index=False))

    # Coverage summary
    print(f"\n=== COVERAGE SUMMARY ===")
    total = len(df_profiles)
    successful = (~df_profiles["error"].astype(bool)).sum()
    print(f"Successful scrapes: {successful}/{total} ({successful/total*100:.0f}%)")
    print(f"With address: {df_profiles['address'].astype(bool).sum()}/{total}")
    print(f"With phone: {df_profiles['phone'].astype(bool).sum()}/{total}")
    print(f"With website: {df_profiles['website'].astype(bool).sum()}/{total}")
    print(f"With tribal council: {df_profiles['tribal_council'].astype(bool).sum()}/{total}")

print("\nData quality review complete.")